# 🌐 Modelos de Lenguaje

**Laboratorio de PLN**
Matías Barreto, 2026
**Encuentro 13 · Bloque 1 — 50 minutos**
* **Alumna:** Dominguez Micaela Belen
---

## Objetivo

Ejecutar un modelo de lenguaje local para entender cómo los LLMs generan texto.

## Al terminar este bloque vas a poder:

1. Cargar y ejecutar Phi-3 en Google Colab usando Hugging Face.
2. Configurar parámetros de inferencia para controlar la salida del modelo.
3. Estructurar prompts en español para distintas tareas.

Lo ejecuto en Kaggle con acelerator GPU T4


## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **LLM (Large Language Model)** | Red neuronal entrenada con enormes cantidades de texto para predecir y generar lenguaje natural. |
| **Inferencia** | El momento en que el modelo ya entrenado recibe un prompt y produce una respuesta. |
| **Phi-3** | Modelo compacto de Microsoft (3.8B parámetros) optimizado para correr en hardware limitado. |
| **Hugging Face Hub** | Repositorio online con miles de modelos listos para descargar y usar. |
| **Tokenizador** | Componente que convierte texto en números que el modelo puede procesar. |

## ⚙️ Configuración del entorno

### Requisitos

| Requisito | Detalle |
|---|---|
| **Google Colab con GPU** | `Entorno de ejecución → Cambiar tipo → GPU T4` |
| **Conexión a Internet** | Para descargar el modelo desde Hugging Face Hub |

La GPU T4 gratuita de Colab es suficiente para Phi-3-mini.

## Instalación de dependencias

- **transformers**: biblioteca de Hugging Face para trabajar con LLMs
- **accelerate**: optimiza la carga e inferencia de modelos grandes

In [1]:
# Instalación de dependencias
# El prefijo ! permite ejecutar comandos de sistema desde el notebook
!pip install transformers>=4.40.1 accelerate>=0.27.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

## ¿Qué es un modelo de lenguaje?

### Analogía

Imaginá un empleado que leyó millones de libros, correos, artículos y conversaciones. Nunca entendió cada texto en profundidad, pero absorbió tantos patrones que puede predecir con alta precisión qué palabra viene después de cualquier secuencia. Eso hace un LLM: predice la siguiente palabra, una y otra vez, hasta generar texto coherente.

### Dónde vive esto en el mundo real

Los LLMs son el motor de ChatGPT, Gemini, Claude y Copilot. Las empresas los usan para atención al cliente, redacción de documentos, análisis de contratos y generación de código. En este curso, Phi-3 va a ser tu primer LLM ejecutable en Colab. En el próximo bloque vas a ver Ollama, que te permite correrlos sin conexión a Internet y sin costo de API.

### ¿Qué es Phi-3?

| Característica | Detalle |
|---|---|
| **Parámetros** | 3.8 mil millones |
| **Contexto máximo** | 4096 tokens (~3000 palabras) |
| **Tipo** | Instruction-tuned (sigue instrucciones) |
| **Idiomas** | Multilingüe, incluye español |
| **Licencia** | Open source (Microsoft) |

### ¿Cómo genera texto?

El modelo opera en tres pasos:

1. **Entrada**: recibe un prompt (texto inicial)
2. **Procesamiento**: analiza el contexto usando su conocimiento interno
3. **Generación**: produce la continuación más probable, token por token

### ✎ Para pensar

- Si el modelo predice la siguiente palabra basándose en patrones estadísticos, ¿por qué puede "inventar" datos que no existen?
- ¿Qué diferencia hay entre un modelo que "sabe" algo y uno que "aprendió a responder" sobre ese algo?

1) porque el modelo no chequea si lo que genera es verdad simplemente elije el tokenn mas probable 
2) y que el llm no tiene idea si es mentira o verdad solo genera la respuesta. Por eso es super importante que se use la ia con total criterio. 

## Carga del modelo

Vamos a cargar Phi-3 en la memoria de la GPU. Este proceso puede tardar 1-2 minutos la primera vez.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargamos el modelo Phi-3-mini
# Este proceso puede tardar 1-2 minutos la primera vez
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",  # Identificador del modelo en Hugging Face
    device_map="cuda",                    # device_map: Dónde cargar el modelo ("cuda" = GPU, "cpu" = procesador)
    torch_dtype="auto",                   # torch_dtype: Tipo de datos numéricos ("auto" selecciona el óptimo)
    trust_remote_code=False,              # trust_remote_code: Por seguridad, no ejecutar código remoto no verificado
)

# Cargamos el tokenizador correspondiente al modelo
# El tokenizador convierte texto en números que el modelo puede procesar
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

print("Modelo cargado exitosamente")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Modelo cargado exitosamente


### Componentes cargados

| Componente | Para qué sirve |
|---|---|
| `AutoModelForCausalLM` | Carga el modelo generativo (predice tokens de izquierda a derecha) |
| `AutoTokenizer` | Convierte texto en números y números en texto |
| `device_map="cuda"` | Carga el modelo en la GPU (mucho más rápido que CPU) |
| `torch_dtype="auto"` | Selecciona automáticamente la precisión numérica óptima |

## Pipeline de generación

Un **pipeline** encapsula en un objeto todos los pasos: tokenización → inferencia → decodificación. Lo usás como una función.

In [3]:
from transformers import pipeline

# Creamos un pipeline de generación de texto
generador = pipeline(
    "text-generation",           # Tipo de tarea
    model=model,                  # Modelo a usar
    tokenizer=tokenizer,          # Tokenizador a usar
    return_full_text=False,       # return_full_text: Si False, solo devuelve el texto generado (no el prompt)
    max_new_tokens=500,           # max_new_tokens: Cantidad máxima de tokens a generar
    do_sample=False               # do_sample: Si False, usa generación determinística (siempre la misma salida)
)

print("Pipeline creado exitosamente")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Pipeline creado exitosamente


### Parámetros clave del pipeline

| Parámetro | Efecto |
|---|---|
| `return_full_text=False` | Devuelve solo el texto generado (sin repetir el prompt) |
| `max_new_tokens=500` | Límite de tokens a generar (más tokens = más tiempo) |
| `do_sample=False` | Generación determinística: siempre elige el token más probable |

### ✎ Para pensar

- Con `do_sample=False`, el modelo siempre da la misma respuesta al mismo prompt. ¿En qué casos preferirías eso? ¿En cuáles no?
- ¿Por qué tiene sentido que `max_new_tokens` afecte el tiempo de ejecución?

1)en un sistema donde quiero que todos los usuarios reciban la misma respuesta a la misma pregunta no para cuando necesito creatividad, como en generacion de texto creativo o cuando quiero variedad en las respuestas.
2)porque el modelo genera token por token, uno a la vez a mas tokens que genere mas pasadas hace por la red neuronal. Si pido 500 tokens tarda el doble que si pido 250.

## Primer ejemplo: generación de texto

El modelo espera mensajes en formato de conversación estructurada:

```python
[{"role": "user", "content": "tu instrucción"}]
```

- `role`: quién habla (`"user"` o `"assistant"`)
- `content`: el texto del mensaje

Este formato permite simular conversaciones de múltiples turnos.

In [4]:
# Definimos el mensaje como una conversación
# Phi-3 está entrenado para recibir mensajes en formato de conversación
mensaje = [
    {"role": "user", "content": "Escribí un párrafo corto sobre el dulce de leche y su importancia en la cultura argentina."}
]

# Generamos la respuesta
salida = generador(mensaje)

# Mostramos el resultado
print(salida[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 El dulce de leche, conocido también como "manjar" en Argentina, es una delicia que ha trascendido fronteras y generaciones. Este dulce, con su sabor rico y sedoso, se ha convertido en un símbolo de la cultura argentina, presente en una variedad de platillos y postres. Desde los tradicionales bizcochos de dulce de leche hasta los famosos "alfajores", este dulce es un ingrediente esencial en la cocina argentina. Su importancia va más allá de la gastronomía; el dulce de leche es un recordatorio de reuniones familiares y celebraciones, un elemento que une a las personas en la convivencia y la tradición.


## Prompt engineering: técnicas básicas

| Técnica | Ejemplo |
|---|---|
| **Sé específico** | "Escribí un email formal..." en lugar de "hablame del mate" |
| **Especificá el formato** | "Listá 5 ventajas usando viñetas" |
| **Incluí contexto** | "Para una audiencia técnica, explicá..." |
| **Pedí extensión** | "En no más de 3 párrafos..." |

In [6]:
# Con temperatura alta - respuestas más creativas y variadas
salida_creativa = generador(
    mensaje,
    do_sample=True,
    temperature=0.9
)
print(salida_creativa[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 El dulce de leche es una delicia icónica en la cultura argentina, remontándose su origen a finales del siglo XIX. Este dulce, único en su sabor suave y ricos sabores tostados, se prepara tradicionalmente a base de leche evaporada e infundida con azúcar. Se ha convertido en un símbolo de los días festivos en Argentina y se puede disfrutar en una variedad de formas, desde postres hasta decoraciones para café y pan. Su fácil digestión y el placer que produce en el paladar han hecho que el dulce de leche sea una parte indispensable de las celebraciones familiares y el ambiente social argentino.


### ✎ Para pensar

- ¿Por qué un prompt más específico genera mejores respuestas? ¿Qué le estás dando al modelo que no tenía antes?
- Probá el mismo prompt con `do_sample=True` y `temperature=0.9`. ¿Qué cambia?

1) Porque le pasas un contexto entonces sabe donde tiene que ir a buscar o sea  hacer una consulta basica deriva en miles de direcciones distintas pero si acoto el contexto el modelo elije sobre las opciones mas relevaNtes.
2) Con twmperatura alta el modelo tomo un camino distinto mas detallado y con dosample en dalse fue a lo mas estandar y tipico. 

In [7]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Probá los siguientes prompts y comparalos con tus compañeros:
#   1. Un texto formal (carta, informe, email académico)
#   2. Un texto creativo (cuento corto, poema, diálogo)
#   3. Un texto técnico (explicación de un concepto, tutorial de un paso)
#
# Bonus: ¿cómo cambia la respuesta si especificás la longitud o el tono?
#
# ─────────────────────────────────────────────────────────────────────────────
mi_prompt = [
    {"role": "user", "content": "Tu instrucción aquí"}
]

mi_salida = generador(mi_prompt)
print(mi_salida[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 La instrucción proporcionada es un marcador de posición que indica que se debe seguir un formato similar al de la instrucción dada anteriormente. No contiene contenido específico, por lo que la respuesta sería simplemente seguir el mismo formato y estructura, pero con contenido relevante para la tarea en cuestión.


Por ejemplo, si la tarea es crear una instrucción para un asistente virtual que genere un poema sobre el atardecer, la instrucción sería:


"Tu instrucción aquí: Genera un poema lírico que capture la belleza y el cambio de las horas del atardecer, utilizando metáforas y una métrica de octosílabos."


texto formal

In [8]:
mi_prompt_1 = [
    {"role": "user", "content": "Escribí un email formal dirigido al rector de una universidad solicitando una beca de mérito académico. Incluí asunto, saludo, cuerpo y despedida. Tono profesional y respetuoso."}
]
mi_salida_1 = generador(mi_prompt_1)
print(mi_salida_1[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Asunto: Solicitud de Beca de Mérito Académico


Estimado Sr. González,


Me dirijo a usted en mi calidad de estudiante de último año de la Facultad de Ciencias de la Universidad de XYZ, con el fin de solicitar una beca de mérito académico que ha sido otorgada anualmente a estudiantes destacados en su disciplina.


Durante los últimos años, he dedicado mi tiempo y esfuerzo a la excelencia académica, lo que me ha permitido obtener calificaciones sobresalientes y participar activamente en proyectos de investigación que han contribuido significativamente a mi campo de estudio. Mi desempeño académico ha sido reconocido por mi asesor, quien me ha recomendado para esta beca.


Adjunto a este correo encontrará mi expediente académico, que incluye certificados de participación en competencias nacionales e internacionales, así como una carta de recomendación de mi asesor.


Confío en que mi trayectoria académica y mi compromiso con el avance de la ciencia justifican mi solicitud. Agradezco de a

texto creativo

In [9]:
mi_prompt_2 = [
    {"role": "user", "content": "Escribí un cuento corto de no más de 3 párrafos sobre una desarrolladora de software argentina que descubre que su código cobró vida propia."}
]
mi_salida_2 = generador(mi_prompt_2)
print(mi_salida_2[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 En la ciudad de Buenos Aires, una desarrolladora de software llamada Valeria trabajaba en la última versión de su aplicación de realidad aumentada. Un día, mientras probaba la aplicación, notó que los objetos virtuales comenzaron a moverse de manera autónoma, interactuando entre sí y con el entorno físico. Valeria, con un ojo crítico y una mente analítica, comenzó a investigar el código.


A medida que se adentraba en el misterio, descubrió que había un error en la lógica de los algoritmos que le permitía a los objetos virtuales aprender y adaptarse a su entorno. Valeria, fascinada por este fenómeno inesperado, decidió explorar este nuevo potencial. Con cada error corregido y cada función mejorada, los objetos virtuales se volvieron más complejos y autónomos.


Valeria, que siempre había buscado la perfección en su código, ahora se enfrentaba a una realidad donde su creación parecía tener una vida propia. La aplicación de realidad aumentada se convirtió en una obra de arte que interac

texto tecnico

In [11]:
mi_prompt_3 = [
    {"role": "user", "content": "Explicá en términos simples qué es un entorno virtual en Python y por qué es importante usarlo. Dirigido a alguien que recién empieza a programar."}
]
mi_salida_3 = generador(mi_prompt_3)
print(mi_salida_3[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Un entorno virtual en Python es como un espacio aislado donde puedes instalar y usar diferentes versiones de Python y paquetes de software sin afectar a otros proyectos o a la versión de Python que tienes instalada en tu computadora. Es importante porque te permite mantener cada proyecto con sus propias dependencias específicas, lo que significa que si un proyecto necesita una versión diferente de un paquete que otro, ambos pueden coexistir sin problemas. Además, al trabajar en un entorno virtual, puedes probar tus cambios sin preocuparte de que afecten a otros proyectos o a tu sistema operativo.


## ⛰️ Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **LLM** | Red neuronal que predice tokens para generar texto coherente |
| **Phi-3** | Modelo compacto (3.8B parámetros) ejecutable en Colab con GPU |
| **Pipeline** | Objeto que encapsula tokenización + inferencia + decodificación |
| **Parámetros** | `max_new_tokens`, `do_sample`, `temperature` controlan la salida |
| **Prompt** | La instrucción en formato `{"role": "user", "content": "..."}` |

**Próximo bloque →** Tokens y Embeddings: cómo el modelo "lee" el texto antes de generarlo, y por qué ese proceso determina qué tan bien entiende el español.